In [ ]:
import os 
import time 
import uuid 
import json
from dotenv import load_dotnev 
from portkey_ai import Portkey,createHeaders, PORTKEY_GATEWAY_URL

In [ ]:
load_dotenv(dotenv_path="../.env")

# Portkey API key — authenticates your app to the Portkey gateway
PORTKEY_API_KEY = os.getenv("PORTKEY_API_KEY", "F43bakYp/Iy5kMTp7w55SSmgRLrp")

# Provider slug — the name you gave when adding Groq in the Portkey dashboard
# Model format: @<slug>/<model-name>
GROQ_SLUG    =  "rag"  # your Groq integration slug
GROQ_MODEL   = f"@{GROQ_SLUG}/llama-3.3-70b-versatile"

# Second Groq integration for multi-provider experiments (5, 6, 10)
# Uses a smaller/faster model as the fallback target
GROQ_SLUG_2      =  "brag"           # your second Groq slug
GROQ_MODEL_SMALL = f"@{GROQ_SLUG_2}/llama-3.1-8b-instant"

# Keep GROQ_API_KEY for the LangChain experiment (Exp 9)
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [ ]:
# helper functions

# section : formats and prints section headings
def section(title):
    print(f"\n{'='*62}")
    print(f"  {title}")
    print(f"{'='*62}")
    
    
# show : formats and displays a question , its answer,execution time and an optional label in a 
#        clean and consistent way

def show(q, answer, ms, label=""):
    bar = chr(9472) * 62
    print(f"\n{bar}")
    print(f"Q: {q}")
    print(f"A: {answer[:260]}{'...' if len(answer) > 260 else ''}")
    note = f" | {label}" if label else ""
    print(f"⏱  {ms:.0f}ms{note}")
    print(bar)

# access the port api key 
portkey = Portkey(api_key=PORTKEY_API_KEY)

# To print the setup details 
print("Setup complete!")
print(f"  Portkey API Key : {'OK' if PORTKEY_API_KEY else 'MISSING'}")
print(f"  Groq slug       : {GROQ_SLUG}")
print(f"  Groq model ref  : {GROQ_MODEL}")
print(f"  Groq slug 2     : {GROQ_SLUG_2}")
print(f"  Small model ref : {GROQ_MODEL_SMALL}")
print(f"\nPortkey Gateway : {PORTKEY_GATEWAY_URL}")

In [ ]:

# BASELINE : direct llm call (No gateway)

# See what a raw LLM calls looks like - no routing , no logging , no resilience


from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

raw_groq = ChatGroq(api_key=GROQ_API_KEY, model="llama-3.3-70b-versatile", temperature=0)

section("BASELINE — Direct Groq Call")

questions = [
    "What is Kubernetes in one sentence?",
    "What is Intel SRIOV?",
]

for q in questions:
    t0 = time.time()  # to start the timing
    r = raw_groq.invoke([HumanMessage(content=q)]) # to get the response from the groq after we invoke it
    show(q, r.content, (time.time()-t0)*1000, label="direct Groq — no gateway")
    # show method is used here to give the response in a organized way

print("\nMissing: logging, retries, fallback, caching, metadata.")

In [ ]:
# EXP 1 : Route through the gateway 

# Instead of directly calling the LLM we will call it through portkey 

section("EXP 1 — Basic Gateway Call")


questions = [
    "What is Kubernetes in one sentence?",
    "What is Intel SRIOV?",
]

for q in questions:
    t0 = time.time()
    r = portkey.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": q}]
    )
    show(q, r.choices[0].message.content, (time.time()-t0)*1000,
         label="routed via Portkey gateway")

print("\n✅ Check portkey.ai → Logs to see both requests fully logged!")
print("   Token count, cost, latency — all tracked. Zero extra code.")

# portkey.chat.completions : This is where the portkey comes into picture 
#        Instead of doing chatgroq.invoke() now we are saying portkey please send this question
#        to the model for me 
# So the flow becomes : application -> portkey gateway -> groq -> response

# Your application is not talking directly to groq anymore 
#    It is talking to portkey 
# Portkey then forwards the request to groq 

# show function here display the question, the answer returned by the LLM , how long it took to get the
#    answer and a note saying the request was routed through the portkey gateway 

# In this experiment we have routed through our gateway , this will take more time than the before 
#     experiment because it went through the gateway 

# We can go to portkeyai and check our logs there 
#      portkeyai website => analytics , logs 
# We can also see the LLM tracing in langsmith , but there we cant see the details about the gateway 

# 1) Portkey(api_key = )  => creates a client that connects the application to the portkey gateway .
#          All LLM requests are sent through this client 

# 2) model="@slug/model-name" : model="@groq/llama-3.3-70b-versatile"
#          This tells Portkey
#          "Please use Groq's Llama 3.3 model for this request."
#    "model='@provider/model-name' tells Portkey which LLM provider and model should handle the 
#         request. Portkey then forwards the request to that provider

# 3) response.choices[0].message.content => Extracts the actual answer generated by the LLM.

In [ ]:
# EXP 2 : Metadata and observability 

# Now we will add metadata also 

section("EXP 2 — Metadata & Observability")

session = str(uuid.uuid4())[:8] # we are creating unique session ids for multiple users

# below are the multiple users
scenarios = [
    ("alice", "enterprise-rag",   "What is Kubernetes RBAC?"),
    ("bob",   "docs-chatbot",     "How does BGP path selection work?"),
    ("carol", "support-bot",      "What is SRIOV virtualization?"),
    ("alice", "enterprise-rag",   "Explain Kubernetes NetworkPolicy"),   # same user, diff Q
] 

for user, feature, q in scenarios:
    t0 = time.time()
    r = portkey.with_options(
        metadata={
            "_user":       user,          # powers per-user analytics in dashboard
            "session_id":  session,
            "feature":     feature,
            "environment": "notebook"
        }
    ).chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": q}]
    )
    ms = (time.time() - t0) * 1000
    print(f"\n👤 {user:8s} | 🔧 {feature:18s} | {ms:.0f}ms")
    print(f"  Q: {q}")
    print(f"  A: {r.choices[0].message.content[:120]}...")

print("\n✅ Dashboard now shows: cost per user, calls per feature, session grouping")

# 1) After scenarios , loops through every scenario one by one 

# 2) portkey.with_options(metadata = { })  => This is where metadata is added 
#           Think of metadata as extra information about the request 
#           Portkey will now know which user asked the question
#    All requests with session ID are grouped together
#    environment = "notebook" => This is where the request came from 

# 3) .chat.completions.create(...) => Portkey sends the request to groq 

# 4) What does portkey dashboard show ?
#     without metadata portkey only knows how many requests we got .
#     with metadata , portkey will know which user sends how many requests (requests user wise)

# 5) In this experiment, we use Portkey's metadata feature to attach additional information 
#    to every LLM request, such as the user, session ID, feature name, and environment. 
#    This metadata is not sent to the LLM for generating the answer; instead, Portkey 
#    stores it for observability. In the Portkey dashboard, we can analyze requests by 
#    user, group them by session, measure costs per feature, and monitor usage across different 
#    environments. This is especially useful in enterprise applications where multiple users 
#    and services share the same AI infrastructure.

# Do developers manually add metadata for every request ?

# 1) No , in production metadata is usually generated automatically . The backend already knows the 
#    logged-in-user, session ID , environment and application feature.
#    A middleware or wrapper function collects this information and attaches it to every portkey 
#    request automatically . 
#    Developers simply call the LLM service without worrying about the metadata.

#    Instead of writing this every time:

#    portkey.with_options(metadata={...}).chat.completions.create(...)

#    companies create a reusable helper:

#    def call_llm(question):
#     metadata = get_request_metadata()   # Automatically fetches user, session, environment
#     return portkey.with_options(metadata=metadata).chat.completions.create(...)

#    Then everywhere else in the code, developers simply write:

#    response = call_llm(question)

#    The metadata is attached automatically behind the scenes. This keeps the code 
#    clean, consistent, and ensures every request is properly tracked
#    without developers repeating the same code.

# Now we can go to portkey api logs and see the requests, there we can see which user sent which  
#     request , how many tokens they have spent , how much money spent 
#     In portkey => analytics, we can track user wise 


In [ ]:
# EXP 3 : Automatic retries 

# Portkey automatically retries failed requests with exponential backoff


retry_config = {
    "retry": {
        "attempts": 3,
        "on_status_codes": [429, 500, 502, 503, 504]
    }
}

portkey_retry = Portkey(api_key=PORTKEY_API_KEY, config=retry_config)


section("EXP 3 — Automatic Retries")
print("Config: 3 retry attempts on [429, 500, 502, 503, 504]")
print("Retries fire automatically on failure — transparent to your code\n")


try:
    t0 = time.time()
    r = portkey_retry.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": "What is a Kubernetes DaemonSet?"}]
    )
    ms = (time.time() - t0) * 1000
    print(f"✅ Succeeded in {ms:.0f}ms")
    print(f"   {r.choices[0].message.content[:300]}")
    print("\nRetry sequence if Groq had failed:")
    print("  Attempt 1 → 429 → wait 1s → Attempt 2 → 429 → wait 2s → Attempt 3")
    print("  Your code only sees the final success or the last failure")
except Exception as e:
    print(f"❌ All attempts failed: {e}")


# 1) This is demonstrating automatic retries .
   
#    If the LLM temporarily fails , portkey will automatically try again without you rewriting the 
#    retry logic .

# 2) retry_config : This is telling portkey , if you receive any of these errors dnt fail immediately.
#                   Try again.
#                   maximum3 attempts and retry only for these error codes.
#                   These are usually temporary errors, so retrying often works.

# 3) portkey_retry : We are creating a portkey client with retry enabled.
#                    So whenever we use this client , apply the retry policy automatically 

# 4) In try block : 
#    In this code , we send a request to the groq model through a portkey client that has retry enabled.
#    If the request succeeds,we calculate the response time and print the models answer. 

#    If groq temporarily fails with retryable errors like 429 or 503, portkey automatically retries
#    the requestup to three times without any extra logic in our application.
   
#    If all retry attempts fail,an exception is raised and handled in the except block. 


In [ ]:
# EXP 4 : Request Timeouts

# This experiment is about Request Timeout 
# Dont wait forever for the LLM.If it doesnt respond within 10 seconds , stop waiting and return 
#     an error.
# This improves the user experience because users arent left waiting indefinitely .

# Config-level timeout (cleanest approach — works inside any routing config)
timeout_config = {"request_timeout": 10000}   # 10 seconds in ms


portkey_timeout = Portkey(api_key=PORTKEY_API_KEY, config=timeout_config)

section("EXP 4 — Request Timeouts")
print("Timeout: 10,000ms (10 seconds). Portkey returns HTTP 408 if exceeded.\n")

try:
    t0 = time.time()
    r = portkey_timeout.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": "Explain Kubernetes networking in 2 sentences."}]
    )
    ms = (time.time() - t0) * 1000
    print(f"✅ Response in {ms:.0f}ms (within 10s timeout)")
    print(f"   {r.choices[0].message.content}")
except Exception as e:
    print(f"⏱  Timed out: {e}")
    print("   Portkey issued a 408. Pair with fallback to auto-switch providers on timeout.")


print("\n--- Combining timeout + retry (production pattern) ---")
combined = {
    "request_timeout": 10000,
    "retry": {"attempts": 2, "on_status_codes": [408, 429, 503]}
}
print(json.dumps(combined, indent=2))


# 1) timeout_config : We are telling the portkey wait for maximum of 10 seconds. If the LLM 
#    doesnt respond by then , stop the request.

#    Since 10000 is in milliseconds => 10000 ms = 10 seconds

# 2) portkey_timeout : This creates a portkey client with timeout enabled. 
#                      Now every request sent using portkey_timeout follows the 10 second timeout rule .

# 3) The try block sends the user request to the groq model through the portkey gateway .
#    Before sending the request , it records the start time . 
#    If the model responds within the configured 10-second timeout, the code calculates the response time 
#    and prints both the execution time and the models answer.

#    If the request exceeds the timeout or any other exception occurs control moves to the except block 
#    where the timeout error is handled gracefully instead of crashing the application .

# 4) This configuration combines a request timeout and automatic retries. Portkey waits
#    up to 10 seconds for the LLM to respond. If the request times out (408), is rate-limited 
#    (429), or the service is temporarily unavailable (503), Portkey automatically retries 
#    the request up to two times. This improves the reliability of the application without
#    requiring any retry logic in our code


# Summary : 
# Exp 4 demonstrates ortkey request timeout feature . We configure a maximum wait time of 10 seconds .
# If the LLM responds within 10 seconds , the response is returned.

# If it takes longer , portkey stops waiting and returns a 408 timeout error .

# The notebook also shows a production best practice of timeout with retires , so temporary timeout or 
# server out errors can be retried automatically before reporting a failure.


In [ ]:
# EXP 5  : Fallbacks
#    If the primary groq model fails , automatically switch to the smaller groq fallback.
#    Users never see the failure.

fallback_config = {
    "strategy": {"mode": "fallback"},
    "targets": [
        {"override_params": {"model": GROQ_MODEL}},     # primary
        {"override_params": {"model": GROQ_MODEL_SMALL}}    # fallback if primary fails
    ]
}

portkey_fallback = Portkey(api_key=PORTKEY_API_KEY, config=fallback_config)


section("EXP 5 — Fallback Routing")
print(f"Strategy: {GROQ_MODEL} → {GROQ_MODEL_SMALL} on failure\n")


fallback_questions = [
    "What is Intel QuickAssist Technology?",
    "Explain Kubernetes persistent volume claims.",
]

for q in fallback_questions:
    try:
        t0 = time.time()
        r = portkey_fallback.chat.completions.create(
            messages=[{"role": "user", "content": q}]
        )
        ms = (time.time() - t0) * 1000
        show(q, r.choices[0].message.content, ms, label="fallback ready")
    except Exception as e:
        print(f"❌ {e}")
        print("   → Check that both GROQ_SLUG and GROQ_SLUG_2 are set correctly in c04")


print("\nFallback chain:")
print("  Groq 2xx   → return immediately")
print("  Groq 4xx/5xx → try small Groq model (8b) automatically")
print("  Check Portkey Logs to see fallback activations")


# 1) fallback_config : You are telling the portkey to use the groq model first . If it fails 
#    automatically try groq_model_small

# 2) portkey_fallback : This creates a protkey client with fallback enabled . 
#    From now on , every request made through this client follows the fallback strategy.

# 3) Every request is first sent to the primary model. If the primary model responds successfully, 
#    the answer is returned immediately. If it fails due to server or availability issues, 
#    Portkey automatically routes the request to the fallback model. This improves application 
#    reliability because users can still receive responses even when the primary model is unavailable.

# 4) Narrow the fallback trigger
#    Default fires on any non-2xx. Narrow it to avoid accidental fallbacks on bad requests:
#    Only fall back on rate limits and server errors

#    "strategy": {"mode": "fallback", "on_status_codes": [429, 503]}

#    Fallback is useful only when the problem is with the LLM or the provider such as the service being 
#    busy(429) or temporarily unavailable (503).
#    If the problem is in our own request, like a bad request (400) or an invalid api key(401)
#    switching to another model will not help because we are sending the same incorrect request . 
   
#    Thats why in production we usually confiure fallback only for temporary errors like 429 or 503.

#    Problem with the model? (busy, server down) → ✅ Fallback
#    Problem with my request? (bad request, invalid API key) → ❌ Don't fallback

In [ ]:
from pydantic._internal._validators import multiple_of_validator
# EXP 6 : Load Balancing 

# 1) We have an application and users use the application , if soo many people are hitting the application 
#    If too many people will hit the application at one it is going to fail

# 2) We have a load balancer => which manages the traffic 
#    We make replicas of the application and we have a load balancer in between 

#    The moment the user query is going to come , they will first hit the load balancer and this load 
#    balancer is going to distribute the traffic.

# 3) In case of LLM also , if too many people are chatting with the chatbot we want to balance our load 
   
#    """A load balancer distributes user requests among multiple servers or LLMs to improve performance, 
#    availability, and reliability."""

load_balance_config = {
    "strategy": {"mode": "loadbalance"},
    "targets": [
        {"override_params": {"model": GROQ_MODEL},   "weight": 0.7},   # 70%
        {"override_params": {"model": GROQ_MODEL_SMALL}, "weight": 0.3}    # 30%
    ]
}

portkey_lb = Portkey(api_key=PORTKEY_API_KEY, config=load_balance_config)

section("EXP 6 — Load Balancing (70% large / 30% small)")


lb_questions = [
    "What is a Kubernetes Ingress resource?",
    "How does OSPF differ from BGP?",
    "What is Intel FPGA acceleration?",
    "Explain Kubernetes HPA.",
    "What is a VLAN trunk?",
    "How does Kubernetes etcd work?",
]

print("Sending 6 requests. Expect ~4 on large model (70b), ~2 on small model (8b) (probabilistic).\n")


for i, q in enumerate(lb_questions, 1):
    try:
        t0 = time.time()
        r = portkey_lb.chat.completions.create(
            messages=[{"role": "user", "content": q}]
        )
        ms = (time.time() - t0) * 1000
        print(f"Req {i} [{ms:.0f}ms]: {q}")
        print(f"         {r.choices[0].message.content[:120]}...")
    except Exception as e:
        print(f"Req {i}: ERROR — {e}")

print("\n✅ Check Portkey Logs to see which provider served each request")
print("   Set weight=0 to pause a target without removing it from the config")

# 1) Instead of sending every request to one model , portkey distributesthe requests between multiple 
#    model based on the weights you configure. 

# 2) load_balance_config =>  This tells portkey : use load balancing Mode 
#    Dnt always use the same model . Share the requests among multiple models . 
   
#    targets => Approximately 70% of the requests should go to GROQ_MODEL model 
#            => Approximately 30% of the requests should go to GROQ_MODEL_SMALL model.

# 3) portkey_lb  => This creates a Portkey client with Load Balancing enabled.

# 4) We have 6 questions, loop sends the 6 questions one by one . 
#    Every request goes to portkey and this portkey decides which model should answer the request 
#    Application doesnt decide that 

#                                Application
#                                     │
#                                     ▼
#                                 Portkey
#                                     │
#                             ┌─────────┴─────────┐
#                             ▼                   ▼
#                     70% Requests          30% Requests
#                         70B Model          8B Model

#         | Fallback                                         | Load Balancing                                  |
# | ------------------------------------------------ | ----------------------------------------------- |
# | Use one model first                              | Use both models from the beginning              |
# | Second model is used **only if the first fails** | Both models actively handle requests            |
# | Purpose: Reliability                             | Purpose: Share workload and improve scalability |

# This experiment demonstrates Portkey's load balancing feature. Two Groq models are configured 
# with weights of 70% and 30%. Every incoming request is routed by Portkey based on these weights, 
# so approximately 70% of the requests are handled by the larger model and 30% by the smaller model.
#  This distributes the workload across multiple models, reduces the chance of overloading a 
#  single model, and improves scalability and overall performance.

# If we see the logs in portkey , we can see both the models in portkey ai => analytics 



In [ ]:
# EXP 7 : Request Caching 

# If the same (or a very similar) question is asked again, portkey returns the previously generated 
#    answer instead of calling the LLM again.

cache_config = {"cache": {"mode": "semantic"}}
# cache_config = {"cache": {"mode": "simple"}} => exact words should match


portkey_cached = Portkey(api_key=PORTKEY_API_KEY, config=cache_config)


section("EXP 7 — Semantic Caching")


# Use a short, precise question with temperature=0 to maximise cache-key stability
q = "Define Kubernetes autoscaling."
call_params = dict(
    model=GROQ_MODEL,
    messages=[{"role": "user", "content": q}],
    temperature=0,
    max_tokens=120,
)

print("--- CALL 1: Cache MISS — Portkey forwards to Groq ---")
t0 = time.time()
r1 = portkey_cached.chat.completions.create(**call_params)
t1 = (time.time() - t0) * 1000
ans1 = r1.choices[0].message.content.strip()
print(f"Answer  : {ans1[:200]}")
print(f"Latency : {t1:.0f}ms | Cost: normal token price")



q = "what is Kubernetes autoscaling."
call_params = dict(
    model=GROQ_MODEL,
    messages=[{"role": "user", "content": q}],
    temperature=0,
    max_tokens=120,
)


print("CALL 2: Same request — should be a Cache HIT")
t0 = time.time()
r2 = portkey_cached.chat.completions.create(**call_params)
t2 = (time.time() - t0) * 1000
ans2 = r2.choices[0].message.content.strip()
print(f"Answer  : {ans2[:200]}")
print(f"Latency : {t2:.0f}ms")


# 1) cache_config => enable semantic caching 
#         => Remember previous questions and answers

# 2) portkey_cached  => Now you have created a portkey client with caching enabled. 
#       => Every request made through this client first checks the cache 

# 3) call_params => This prepares the request . 
#        temperature = 0 
#        Because temperature 0 makes the model deterministic.
#        The same question is much more likely to produce the same answer, making cache more reliable.

# 4) call 1 => This is the first time you are asking the question . Since the portkey as never seen it 
#               before , the cache is empty 
#               This is called a "cache miss"

#     application => portkey cache => question found => No => Groq => Generate answer 
#       => Save answer in cache => Return answer 

#     portkey_cached.chat.completions.create(**call_params) => Since this is the first request ,
#     portkey forwards it to groq 
#     Groq generates answer

#     ans2 = r2.choices[0].message.content.strip() => Extracts the answer

# 5) The call 1 was a cache miss , call 2 should be a cache hit . 
   
#    In call 2 : We are asking the same question again 
#    Since semantic caching is enabled , portkey first checks its cache and finds that it already has a 
#    stored response for this question .

#    Instead of forwarding the request to the LLM it immediately returns the cached answer. 
#    This results in much lower latency and avoids additional token usage and API cost.

In [ ]:
# EXP 8 : Integrating with Langchain 

# Earlier we are using chatgroq , now we are using chatopenai 
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Portkey exposes an OpenAI-compatible endpoint at PORTKEY_GATEWAY_URL
# Pass PORTKEY_API_KEY as the api_key; the @slug/model handles provider routing
portkey_llm = ChatOpenAI(
    api_key=PORTKEY_API_KEY,          # Portkey API key (not Groq key)
    base_url=PORTKEY_GATEWAY_URL,     # Portkey gateway endpoint
    model=GROQ_MODEL,                 # "@rag/llama-3.3-70b-versatile"
    temperature=0,
    default_headers=createHeaders(    # adds x-portkey-* headers
        api_key=PORTKEY_API_KEY,
        metadata={
            "_user":       "rag-pipeline",
            "environment": "notebook",
            "feature":     "langchain-integration"
        }
    )
)

section("EXP 9 — LangChain Drop-in")

# Test 1: Direct invoke (like app/agents/nodes/planner.py)
print("--- Test 1: Direct invoke (like planner node) ---")
t0 = time.time()
r = portkey_llm.invoke([
    SystemMessage(content="You are an Enterprise IT Assistant."),
    HumanMessage(content="What is the difference between a Deployment and a StatefulSet?")
])
ms = (time.time() - t0) * 1000
print(f"✅ {ms:.0f}ms")
print(r.content[:250])

# Test 2: LCEL chain (like app/agents/nodes/responder.py)
print("\n--- Test 2: LCEL chain (like responder node) ---")
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an Enterprise IT expert. Be concise."),
    ("human",  "{question}")
])
chain = prompt | portkey_llm | StrOutputParser()

t0 = time.time()
answer = chain.invoke({"question": "Explain Kubernetes pod affinity rules."})
ms = (time.time() - t0) * 1000
print(f"✅ {ms:.0f}ms")
print(answer[:250])

print("\n→ This is a drop-in for ChatGroq in app/agents/nodes/*.py")
print("→ Every RAG pipeline call is now logged in Portkey with metadata")


# 1) The goal of this experiment is not to introduce a new portkey feature. 
#    Instead it shows , 
#    How can we use portkey inside a langchain application without changing much of our existing code.

#    "Why chatopenai and not chatgroq ?"
#    Every LLM companys api has an address(URL):
#      Groqs address : https://api.groq.com/openai/v1 => This is the url on which we hit the groq client 
#      portkeys address : https://api.portkey.ai/v1 => This is the url on which the portkey client   

#    When we are using chatgroq we are hitting the groq client internally and above one is the url ,
#    now this url is hardcoded inside chatgroq library of langchain.
#    But when you use chatopenai , where it gives us a way to pass base_url , using this base_url method
#    we can directly hit any particular api or any client with this base_url 

#    chatopenai is just a standard way of using a langchain interface and using the base_url parameter 
#    we can manipulate to which client we are hitting on. In our scenario we are hitting on 
#    portkey client .
#    We cant hit that with chatgroq because inside chatgroq url , it is hardcoded 
   
#    "Summary"  : We use chatopenai instead of chatgroq because chatopenai allows us to specify a 
#    base_url . We point that base_url to the portkey gateway , so all langchain requests first go to portkey 
#    Portkey then routes them to the configured provider such as Groq, openai , gemini and claude. 
#     Whereas "chatgroq" is designed to communicate directly with groq api and doesnot provide this 
#     routing facility .

#     In earlier experiments of 1-8  our goal is to learn portkey features so we have used "portkey SDK"  
#     from portkey_ai import Portkey => This is the portkey SDK
#     so here there was no need for chatopenai or chatgroq. The portkey SDK itself handled everything 
#     o demonstrate Portkey features such as retry, fallback, caching, and load balancing.

#     In experiment 9 , the goal was different we want to integrate portkey into langchain application.
#     LangChain expects an LLM implementation like ChatOpenAI, so we configure ChatOpenAI to point 
#     to the Portkey Gateway using base_url. This lets existing LangChain code work with 
#     Portkey without changing the application's logic.

#     "Can I use ChatGroq with the Portkey Gateway?"

#     Not with the standard LangChain ChatGroq integration. ChatGroq is designed to communicate 
#     directly with the Groq API. To route requests through the Portkey Gateway, we use 
#     ChatOpenAI because it supports a configurable base_url. By pointing base_url to 
#     the Portkey Gateway, every LangChain request goes through Portkey, which then forwards 
#     it to the configured provider such as Groq.



# 2) portkey_llm => This code creates a langchain LLM object configured to use the portkey gateway .
   
#    api_key : This tells the portkey "I am an authorized user"
#              Without this key , portkey wont accept your request
   
#    base_url : Normally langchain sends requests directly to the AI provider. 
#               Here we are saying : Don't send requests directly to any AI provider. Send everything 
#                                     to Portkey first.
   
#    model : Now you're telling Portkey :  After you receive my request, send it to this model.
#            Then portkey forwards the requests to Groq 

#    default_headers : adds extra information to every request.
#                      Metadata is just an extra information . It is not sent to the AI model
#                      It is only stored in portkey logs.

# 3) Test 1 :
#    This code demonstrates a direct LangChain LLM call using invoke(). We provide a SystemMessage 
#    to define the AI's role and a HumanMessage containing the user's question. 
#    LangChain sends both messages through the Portkey Gateway to the configured model 
#    (Groq in this case). The response time is measured, and the generated answer is printed 
#    using r.content.

# 4) Test 2 : 
#    In this example, we first create a reusable prompt template. Then we connect it to the 
#    Portkey LLM and an output parser to form a chain. Whenever we call chain.invoke(), 
#    the question is inserted into the template, sent to the AI through Portkey, and the final 
#    text answer is returned. This makes the code reusable because we only need to change the
#    question each time.

#     | Test 1             | Test 2                                    |
#     | ------------------ | ----------------------------------------- |
#     | Direct `invoke()`  | LCEL Chain                                |
#     | No prompt template | Uses a reusable prompt template           |
#     | No parser          | Uses `StrOutputParser()`                  |
#     | Simple LLM call    | Complete workflow (Prompt → LLM → Parser) |
